Caso 2. Imagen de medio grafico color
En este caso el problema puede estar en la geometria, en el contraste, en el color o en danos localizados. Antes de intervenir, conviene distinguir si el principal obstaculo es la perspectiva, la cromaticidad, la iluminacion o la presencia de manchas.

Sugerencias tecnicas posibles: rectificacion de perspectiva, mejora sobre HSV o LAB, CLAHE, suavizado e inpainting si hubiera dano localizado.

In [ ]:
nombre_imagen_medio_color = "IMG_5426.jpeg"

imagen_medio_color_rgb = None

if nombre_imagen_medio_color != "":
    ruta_medio_color = carpeta_imagenes_tfi / nombre_imagen_medio_color
    imagen_medio_color_rgb = cargar_rgb(ruta_medio_color)
    mostrar_imagen(imagen_medio_color_rgb, "Medio grafico color: imagen original")
    mostrar_canales_rgb(imagen_medio_color_rgb)

diagnostico_medio_color = (
    "La imagen es la tapa de un libro fotografiada con luz ambiente cálida. "
    "Presenta una dominante amarillenta que apaga los colores originales: "
    "el naranja pierde viveza y el azul/violeta aparece desaturado. "
    "Además se observan rayaduras blancas localizadas sobre el fondo negro "
    "y una leve pérdida de contraste general por la iluminación indirecta."
)

objetivo_medio_color = (
    "Recuperar la saturación y el balance de color de la tapa, corrigiendo "
    "la dominante cálida para que el naranja y el azul se acerquen a sus "
    "valores originales de impresión."
)

print("Diagnostico inicial:", diagnostico_medio_color)
print("Objetivo de mejora:", objetivo_medio_color)

In [ ]:
imagen_medio_color_estrategia_1 = None

if imagen_medio_color_rgb is not None:
    imagen_medio_color_estrategia_1 = imagen_medio_color_rgb.copy()

    # Convertimos a HSV para manipular saturación e intensidad por separado.
    # En HSV el canal S controla la viveza del color sin afectar el tono (H).
    hsv = cv2.cvtColor(imagen_medio_color_estrategia_1, cv2.COLOR_RGB2HSV).astype("float32")

    # Subimos la saturación un 40%. Si el valor supera 255 lo recortamos.
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.4, 0, 255)

    # También subimos levemente el brillo para compensar la luz ambiente oscura.
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * 1.1, 0, 255)

    imagen_medio_color_estrategia_1 = cv2.cvtColor(
        hsv.astype("uint8"), cv2.COLOR_HSV2RGB
    )

    mostrar_comparacion(
        imagen_medio_color_rgb,
        imagen_medio_color_estrategia_1,
        "Original",
        "Medio color: estrategia 1"
    )

observacion_medio_color_estrategia_1 = (
    "La mejora en HSV incrementa la saturación de todos los colores por igual. "
    "El naranja y el azul ganan viveza, pero la dominante cálida no se corrige: "
    "como el tono (H) no se toca, el sesgo amarillento de la iluminación se mantiene. "
    "El resultado es más vistoso pero no más fiel al color original de impresión."
)
print(observacion_medio_color_estrategia_1)

In [ ]:
imagen_medio_color_estrategia_2 = None

if imagen_medio_color_rgb is not None:
    imagen_medio_color_estrategia_2 = imagen_medio_color_rgb.copy()

    # Convertimos a LAB. En este espacio:
    # L = luminosidad, A = eje verde-rojo, B = eje azul-amarillo.
    # La dominante cálida se manifiesta como exceso en el canal B (más amarillo).
    lab = cv2.cvtColor(imagen_medio_color_estrategia_2, cv2.COLOR_RGB2LAB).astype("float32")

    # Aplicamos CLAHE sobre L para mejorar contraste sin saturar zonas ya claras.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0].astype("uint8"))

    # Corregimos la dominante restando del canal B (reducimos el exceso de amarillo).
    # El valor 8 fue elegido observando que la dominante es moderada, no extrema.
    lab[:, :, 2] = np.clip(lab[:, :, 2] - 8, 0, 255)

    imagen_medio_color_estrategia_2 = cv2.cvtColor(
        lab.astype("uint8"), cv2.COLOR_LAB2RGB
    )

    mostrar_comparacion(
        imagen_medio_color_rgb,
        imagen_medio_color_estrategia_2,
        "Original",
        "Medio color: estrategia 2"
    )

observacion_medio_color_estrategia_2 = (
    "La corrección en LAB actúa sobre dos problemas por separado: "
    "CLAHE mejora el contraste en L sin afectar el color, y restar del canal B "
    "reduce la dominante amarillenta de la luz ambiente. "
    "El azul/violeta del personaje y el naranja del fondo quedan más equilibrados "
    "entre sí. A diferencia de HSV, esta estrategia corrige el sesgo de color "
    "además de mejorar la viveza general."
)
print(observacion_medio_color_estrategia_2)

In [ ]:
# Elegimos la estrategia 2 como versión final.
imagen_medio_color_final = imagen_medio_color_estrategia_2

justificacion_medio_color_final = (
    "Se elige la estrategia 2 (LAB + CLAHE) porque aborda el problema principal: "
    "la dominante cálida de la iluminación. HSV mejora la saturación global pero "
    "no corrige el sesgo de color, mientras que operar en LAB permite separar "
    "luminosidad y crominancia y actuar sobre cada uno con precisión. "
    "Límite del método: las rayaduras blancas sobre el fondo negro no se corrigen "
    "en ninguna de las dos estrategias, ya que requieren inpainting localizado "
    "y están fuera del alcance de una mejora global de color."
)
print(justificacion_medio_color_final)

if imagen_medio_color_final is not None:
    ruta_salida_medio_color = carpeta_salidas_tfi / "medio_grafico_color_final.png"
    guardar_rgb(ruta_salida_medio_color, imagen_medio_color_final)

    if imagen_medio_color_rgb is not None:
        mostrar_comparacion(
            imagen_medio_color_rgb,
            imagen_medio_color_final,
            "Medio color: antes",
            "Medio color: version final"
        )

    print(f"Salida guardada en: {ruta_salida_medio_color.resolve()}")

Caso 3. Imagen de medio grafico blanco/negro
En este caso suele importar la legibilidad. El objetivo es separar de manera razonable el fondo y los trazos, sin borrar informacion importante. Por eso conviene pensar si necesitas trabajar en grises, suavizar, umbralizar y limpiar la mascara resultante.

Sugerencias tecnicas posibles: conversion a grises, suavizado, umbral global, Otsu, adaptiveThreshold, apertura o clausura morfologica.

In [ ]:
nombre_imagen_medio_bn = "Imagen.PNG"

imagen_medio_bn_gris = None

if nombre_imagen_medio_bn != "":
    ruta_medio_bn = carpeta_imagenes_tfi / nombre_imagen_medio_bn
    imagen_medio_bn_gris = cargar_gris(ruta_medio_bn)
    mostrar_imagen(imagen_medio_bn_gris, "Medio grafico blanco/negro: imagen original")
    mostrar_histograma_gris(imagen_medio_bn_gris)

diagnostico_medio_bn = (
    "La imagen es una página de diario fotografiada con luz ambiente. "
    "Presenta iluminación desigual (zona izquierda más oscura que el centro), "
    "fondo gris por el papel de diario (no blanco puro), dominante de color cálido "
    "y mezcla de texto de distintos tamaños con fotografías. "
    "Un umbral global fallará porque el valor de fondo varía a lo largo de la imagen."
)

objetivo_medio_bn = (
    "Binarizar la página para separar texto e imágenes del fondo gris, "
    "recuperando legibilidad en toda la hoja a pesar de la iluminación desigual."
)

print("Diagnostico inicial:", diagnostico_medio_bn)
print("Objetivo de mejora:", objetivo_medio_bn)

In [ ]:
imagen_medio_bn_estrategia_1 = None

if imagen_medio_bn_gris is not None:
    imagen_medio_bn_estrategia_1 = imagen_medio_bn_gris.copy()

    # Suavizado gaussiano leve para reducir ruido antes de umbralizar
    suavizada_1 = cv2.GaussianBlur(imagen_medio_bn_estrategia_1, (5, 5), 0)

    # Otsu calcula automáticamente el umbral óptimo buscando el mejor corte bimodal.
    # En una página con iluminación desigual, ese único umbral global
    # no puede adaptarse a zonas más oscuras y más claras al mismo tiempo.
    _, imagen_medio_bn_estrategia_1 = cv2.threshold(
        suavizada_1, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    mostrar_comparacion(
        imagen_medio_bn_gris,
        imagen_medio_bn_estrategia_1,
        "Original",
        "Medio blanco/negro: estrategia 1"
    )

observacion_medio_bn_estrategia_1 = (
    "Otsu encuentra un único umbral para toda la imagen. "
    "En la zona central más iluminada el texto queda razonablemente legible, "
    "pero en la zona izquierda más oscura el umbral global es demasiado alto: "
    "parte del texto se pierde porque su gris no supera el corte calculado. "
    "El método no maneja la variación de iluminación a lo largo de la página."
)
print(observacion_medio_bn_estrategia_1)

In [ ]:
imagen_medio_bn_estrategia_2 = None

if imagen_medio_bn_gris is not None:
    imagen_medio_bn_estrategia_2 = imagen_medio_bn_gris.copy()

    # Mediana: más robusta que gaussiano para preservar bordes de letras
    suavizada_2 = cv2.medianBlur(imagen_medio_bn_estrategia_2, 3)

    # adaptiveThreshold calcula un umbral distinto por región local.
    # blockSize=61: ventana grande porque la variación de luz es gradual y de escala amplia.
    # C=10: compensación para no clasificar el fondo gris del papel como texto.
    imagen_medio_bn_estrategia_2 = cv2.adaptiveThreshold(
        suavizada_2,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY,
        blockSize=61,
        C=10
    )

    # Apertura: elimina puntos blancos aislados (ruido del papel y la trama de impresión)
    # Cierre: une pequeñas interrupciones dentro de los trazos de las letras
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    imagen_medio_bn_estrategia_2 = cv2.morphologyEx(
        imagen_medio_bn_estrategia_2, cv2.MORPH_OPEN, kernel
    )
    imagen_medio_bn_estrategia_2 = cv2.morphologyEx(
        imagen_medio_bn_estrategia_2, cv2.MORPH_CLOSE, kernel
    )

    mostrar_comparacion(
        imagen_medio_bn_gris,
        imagen_medio_bn_estrategia_2,
        "Original",
        "Medio blanco/negro: estrategia 2"
    )

observacion_medio_bn_estrategia_2 = (
    "El umbral adaptativo computa un valor de corte distinto para cada zona, "
    "lo que le permite responder a la iluminación desigual de la fotografía. "
    "El texto queda legible tanto en la zona más oscura como en la más clara. "
    "La limpieza morfológica posterior elimina el ruido de trama del papel de diario "
    "y une letras con pequeñas interrupciones. El resultado es más homogéneo "
    "que Otsu en toda la extensión de la página."
)
print(observacion_medio_bn_estrategia_2)

In [ ]:
# Elegimos la estrategia 2 como versión final.
imagen_medio_bn_final = imagen_medio_bn_estrategia_2

justificacion_medio_bn_final = (
    "Se elige la estrategia 2 (adaptiveThreshold + morfología) porque resuelve "
    "el problema central: la iluminación desigual de la foto. "
    "Otsu asigna un único umbral que beneficia las zonas bien iluminadas "
    "pero pierde texto en las más oscuras. El umbral adaptativo con blockSize=61 "
    "compensa esa variación localmente, manteniendo legibilidad en toda la página. "
    "Límite del método: las zonas de las fotografías impresas dentro del diario "
    "se binarizarán con pérdida de detalle, ya que no son texto y no se benefician "
    "del mismo criterio de separación."
)
print(justificacion_medio_bn_final)

if imagen_medio_bn_final is not None:
    ruta_salida_medio_bn = carpeta_salidas_tfi / "medio_grafico_bn_final.png"
    guardar_gris(ruta_salida_medio_bn, imagen_medio_bn_final)

    if imagen_medio_bn_gris is not None:
        mostrar_comparacion(
            imagen_medio_bn_gris,
            imagen_medio_bn_final,
            "Medio blanco/negro: antes",
            "Medio blanco/negro: version final"
        )

    print(f"Salida guardada en: {ruta_salida_medio_bn.resolve()}")